This file is try to check out the ViT model for CLIP.

Source from: https://www.geeksforgeeks.org/computer-vision/vision-transformers-vit-in-image-recognition/

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from timm import create_model

In [11]:
# Define Transformations for the dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resizing images to 224x224 as ViT expects larger images
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Take 10% data
train_size = int(0.02 * len(train_dataset))
test_size = int(0.02 * len(test_dataset))

train_dataset = Subset(train_dataset, range(train_size))
test_dataset = Subset(test_dataset, range(test_size))

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# Define Vision Transformer Model
model = create_model('vit_base_patch16_224', pretrained=True, num_classes=10)  # Using a pretrained ViT model
model = model.to(device)

# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

Using device: cpu


In [7]:
def train_model(model, train_loader, criterion, optimizer, device, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct, total = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {running_loss / len(train_loader)}, Accuracy: {100 * correct / total:.2f}%')

In [9]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Accuracy on test data: {100 * correct / total:.2f}%')

In [ ]:
# Train and Evaluate the Model
train_model(model, train_loader, criterion, optimizer, device, epochs=1)
evaluate_model(model, test_loader, device)